# Batch processing with the Batch API

The new Batch API allows to **create async batch jobs for a lower price and with higher rate limits**.

Batches will be completed within 24h, but may be processed sooner depending on global usage. 

Ideal use cases for the Batch API include:

- Tagging, captioning, or enriching content on a marketplace or blog
- Categorizing and suggesting answers for support tickets
- Performing sentiment analysis on large datasets of customer feedback
- Generating summaries or translations for collections of documents or articles

and much more!

This cookbook will walk you through how to use the Batch API with a couple of practical examples.

We will start with an example to categorize movies using `gpt-4o-mini`, and then cover how we can use the vision capabilities of this model to caption images.

Please note that multiple models are available through the Batch API, and that you can use the same parameters in your Batch API calls as with the Chat Completions endpoint.

## Setup

In [3]:
# Make sure you have the latest version of the SDK available to use the Batch API
%pip install openai --upgrade

Note: you may need to restart the kernel to use updated packages.


In [4]:
import json
from openai import OpenAI
import pandas as pd
from IPython.display import Image, display

## First example: Categorizing movies

In this example, we will use `gpt-4o-mini` to extract movie categories from a description of the movie. We will also extract a 1-sentence summary from this description. 

We will use [JSON mode](https://platform.openai.com/docs/guides/text-generation/json-mode) to extract categories as an array of strings and the 1-sentence summary in a structured format. 

For each movie, we want to get a result that looks like this:

```
{
    categories: ['category1', 'category2', 'category3'],
    summary: '1-sentence summary'
}
```

### Loading data

We will use the IMDB top 1000 movies dataset for this example. 

In [20]:
dataset_path = "data/imdb_top_1000.csv"

df = pd.read_csv(dataset_path)
df.head()

,Poster_Link,Series_Title,Released_Year,Certificate,Runtime,Genre,IMDB_Rating,Overview,Meta_score,Director,Star1,Star2,Star3,Star4,No_of_Votes,Gross
0,https://m.media-amazon.com/images/M/MV5BMDFkYT...,The Shawshank Redemption,1994,A,142 min,Drama,9.3,Two imprisoned men bond over a number of years...,80.0,Frank Darabont,Tim Robbins,Morgan Freeman,Bob Gunton,William Sadler,2343110,"28,341,469"
1,https://m.media-amazon.com/images/M/MV5BM2MyNj...,The Godfather,1972,A,175 min,"Crime, Drama",9.2,An organized crime dynasty's aging patriarch t...,100.0,Francis Ford Coppola,Marlon Brando,Al Pacino,James Caan,Diane Keaton,1620367,"134,966,411"
2,https://m.media-amazon.com/images/M/MV5BMTMxNT...,The Dark Knight,2008,UA,152 min,"Action, Crime, Drama",9.0,When the menace known as the Joker wreaks havo...,84.0,Christopher Nolan,Christian Bale,Heath Ledger,Aaron Eckhart,Michael Caine,2303232,"534,858,444"
3,https://m.media-amazon.com/images/M/MV5BMWMwMG...,The Godfather: Part II,1974,A,202 min,"Crime, Drama",9.0,The early life and career of Vito Corleone in ...,90.0,Francis Ford Coppola,Al Pacino,Robert De Niro,Robert Duvall,Diane Keaton,1129952,"57,300,000"
4,https://m.media-amazon.com/images/M/MV5BMWU4N2...,12 Angry Men,1957,U,96 min,"Crime, Drama",9.0,A jury holdout attempts to prevent a miscarria...,96.0,Sidney Lumet,Henry Fonda,Lee J. Cobb,Martin Balsam,John Fiedler,689845,"4,360,000"


### Processing step 

Here, we will prepare our requests by first trying them out with the Chat Completions endpoint.

Once we're happy with the results, we can move on to creating the batch file.

In [ ]:
import requests
import json

categorize_system_prompt = '''
Your goal is to extract movie categories from movie descriptions, as well as a 1-sentence summary for these movies.
You will be provided with a movie description, and you will output a json object containing the following information:

{
    categories: string[] // Array of categories based on the movie description,
    summary: string // 1-sentence summary of the movie based on the movie description
}

Categories refer to the genre or type of the movie, like "action", "romance", "comedy", etc. Keep category names simple and use only lower case letters.
Movies can have several categories, but try to keep it under 3-4. Only mention the categories that are the most obvious based on the description.
'''

def get_categories(description):
    prompt = f"{categorize_system_prompt}\n\nMovie description:\n{description}\n\nRespond with the JSON only."

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "phi",
            "prompt": prompt,
            "temperature": 0.1,
            "stream": False
        }
    )
    response.raise_for_status()
    content = response.json().get("response", "")

    try:
        # Attempt to parse the returned JSON string to Python dict
        result = json.loads(content)
    except json.JSONDecodeError:
        print("Warning: Failed to parse JSON response:")
        print(content)
        result = None

    return result


In [ ]:
import requests

def get_categories(description):
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": "phi",
        "prompt": description,
        "stream": False
    }
    
    response = requests.post(url, json=payload)
    response.raise_for_status()  # raise error if bad status
    
    data = response.json()
    return data.get("response", "")
description = "Your movie description or text here"
result = get_categories(description)
print(result)


### Creating the batch file

The batch file, in the `jsonl` format, should contain one line (json object) per request.
Each request is defined as such:

```
{
    "custom_id": <REQUEST_ID>,
    "method": "POST",
    "url": "/v1/chat/completions",
    "body": {
        "model": <MODEL>,
        "messages": <MESSAGES>,
        // other parameters
    }
}
```

Note: the request ID should be unique per batch. This is what you can use to match results to the initial input files, as requests will not be returned in the same order.

In [ ]:
# Creating an array of json tasks

tasks = []

for index, row in df.iterrows():
    
    description = row['Overview']
    
    task = {
        "custom_id": f"task-{index}",
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            # This is what you would have in your Chat Completions API call
            "model": "gpt-4o-mini",
            "temperature": 0.1,
            "response_format": { 
                "type": "json_object"
            },
            "messages": [
                {
                    "role": "system",
                    "content": categorize_system_prompt
                },
                {
                    "role": "user",
                    "content": description
                }
            ],
        }
    }
    
    tasks.append(task)

In [ ]:
# Creating the file

file_name = "data/batch_tasks_movies.jsonl"

with open(file_name, 'w') as file:
    for obj in tasks:
        file.write(json.dumps(obj) + '\n')

### Uploading the file

In [ ]:
import requests

with open(file_name, "r", encoding="utf-8") as f:
    file_content = f.read()

response = requests.post("http://localhost:11434/api/generate", json={
    "model": "phi",
    "prompt": file_content,
    "stream": False
})

response.raise_for_status()
result = response.json().get("response", "")


In [ ]:
print(result)

### Creating the batch job

In [36]:
# Instead of creating a batch job, do this:

import requests

def run_batch_job(prompt_list):
    results = []
    for prompt in prompt_list:
        response = requests.post("http://localhost:11434/api/generate", json={
            "model": "phi",
            "prompt": prompt,
            "stream": False
        })
        response.raise_for_status()
        results.append(response.json().get("response", ""))
    return results

# Usage example:


### Checking batch status

Note: this can take up to 24h, but it will usually be completed faster.

You can continue checking until the status is 'completed'.

In [38]:
prompts = [ "Prompt 1 here", "Prompt 2 here", "Prompt 3 here" ]
batch_results = run_batch_job(prompts)

for res in batch_results:
    print(res)


### Retrieving results

In [40]:
import requests

response = requests.post("http://localhost:11434/api/generate", json={
    "model": "phi",
    "prompt": "Your prompt here",
    "stream": False
})
response.raise_for_status()

result = response.json().get("response", "")

# Save result to a local file if needed
with open("result.txt", "w", encoding="utf-8") as f:
    f.write(result)

print(result)


In [41]:
result_file_name = "data/batch_job_results_movies.jsonl"

with open(result_file_name, 'w', encoding='utf-8') as file:
    file.write(result)


In [42]:
import json

result_file_name = "data/batch_job_results_movies.jsonl"

results = []
with open(result_file_name, 'r', encoding='utf-8') as file:
    for line in file:
        line = line.strip()
        if line:  # skip empty lines
            json_object = json.loads(line)
            results.append(json_object)

# Now `results` is a list of dicts loaded from your .jsonl file


### Reading results
Reminder: the results are not in the same order as in the input file.
Make sure to check the custom_id to match the results against the input requests

In [44]:
# Reading only the first results
for res in results[:5]:
    task_id = res['custom_id']
    # Getting index from task id
    index = task_id.split('-')[-1]
    result = res['response']['body']['choices'][0]['message']['content']
    movie = df.iloc[int(index)]
    description = movie['Overview']
    title = movie['Series_Title']
    print(f"TITLE: {title}\nOVERVIEW: {description}\n\nRESULT: {result}")
    print("\n\n----------------------------\n\n")

## Second example: Captioning images

In this example, we will use `gpt-4-turbo` to caption images of furniture items. 

We will use the vision capabilities of the model to analyze the images and generate the captions.

### Loading data

We will use the Amazon furniture dataset for this example.

In [47]:
dataset_path = "data/amazon_furniture_dataset.csv"
df = pd.read_csv(dataset_path)
df.head()

,asin,url,title,brand,price,availability,categories,primary_image,images,upc,...,color,material,style,important_information,product_overview,about_item,description,specifications,uniq_id,scraped_at
0,B0CJHKVG6P,https://www.amazon.com/dp/B0CJHKVG6P,"GOYMFK 1pc Free Standing Shoe Rack, Multi-laye...",GOYMFK,$24.99,Only 13 left in stock - order soon.,"['Home & Kitchen', 'Storage & Organization', '...",https://m.media-amazon.com/images/I/416WaLx10j...,['https://m.media-amazon.com/images/I/416WaLx1...,NaN,...,White,Metal,Modern,[],"[{'Brand': ' GOYMFK '}, {'Color': ' White '}, ...",['Multiple layers: Provides ample storage spac...,"multiple shoes, coats, hats, and other items E...","['Brand: GOYMFK', 'Color: White', 'Material: M...",02593e81-5c09-5069-8516-b0b29f439ded,2024-02-02 15:15:08
1,B0B66QHB23,https://www.amazon.com/dp/B0B66QHB23,"subrtex Leather ding Room, Dining Chairs Set o...",subrtex,NaN,NaN,"['Home & Kitchen', 'Furniture', 'Dining Room F...",https://m.media-amazon.com/images/I/31SejUEWY7...,['https://m.media-amazon.com/images/I/31SejUEW...,NaN,...,Black,Sponge,Black Rubber Wood,[],NaN,['【Easy Assembly】: Set of 2 dining room chairs...,subrtex Dining chairs Set of 2,"['Brand: subrtex', 'Color: Black', 'Product Di...",5938d217-b8c5-5d3e-b1cf-e28e340f292e,2024-02-02 15:15:09
2,B0BXRTWLYK,https://www.amazon.com/dp/B0BXRTWLYK,Plant Repotting Mat MUYETOL Waterproof Transpl...,MUYETOL,$5.98,In Stock,"['Patio, Lawn & Garden', 'Outdoor Décor', 'Doo...",https://m.media-amazon.com/images/I/41RgefVq70...,['https://m.media-amazon.com/images/I/41RgefVq...,NaN,...,Green,Polyethylene,Modern,[],"[{'Brand': ' MUYETOL '}, {'Size': ' 26.8*26.8 ...","['PLANT REPOTTING MAT SIZE: 26.8"" x 26.8"", squ...",NaN,"['Brand: MUYETOL', 'Size: 26.8*26.8', 'Item We...",b2ede786-3f51-5a45-9a5b-bcf856958cd8,2024-02-02 15:15:09
3,B0C1MRB2M8,https://www.amazon.com/dp/B0C1MRB2M8,"Pickleball Doormat, Welcome Doormat Absorbent ...",VEWETOL,$13.99,Only 10 left in stock - order soon.,"['Patio, Lawn & Garden', 'Outdoor Décor', 'Doo...",https://m.media-amazon.com/images/I/61vz1Igler...,['https://m.media-amazon.com/images/I/61vz1Igl...,NaN,...,A5589,Rubber,Modern,[],"[{'Brand': ' VEWETOL '}, {'Size': ' 16*24INCH ...","['Specifications: 16x24 Inch ', "" High-Quality...",The decorative doormat features a subtle textu...,"['Brand: VEWETOL', 'Size: 16*24INCH', 'Materia...",8fd9377b-cfa6-5f10-835c-6b8eca2816b5,2024-02-02 15:15:10
4,B0CG1N9QRC,https://www.amazon.com/dp/B0CG1N9QRC,JOIN IRON Foldable TV Trays for Eating Set of ...,JOIN IRON Store,$89.99,Usually ships within 5 to 6 weeks,"['Home & Kitchen', 'Furniture', 'Game & Recrea...",https://m.media-amazon.com/images/I/41p4d4VJnN...,['https://m.media-amazon.com/images/I/41p4d4VJ...,NaN,...,Grey Set of 4,Iron,X Classic Style,[],NaN,['Includes 4 Folding Tv Tray Tables And one Co...,Set of Four Folding Trays With Matching Storag...,"['Brand: JOIN IRON', 'Shape: Rectangular', 'In...",bdc9aa30-9439-50dc-8e89-213ea211d66a,2024-02-02 15:15:11


### Processing step 

Again, we will first prepare our requests with the Chat Completions endpoint, and create the batch file afterwards.

In [67]:
import requests
import json

categorize_system_prompt = '''
Your goal is to extract movie categories from movie descriptions, as well as a 1-sentence summary for these movies.
You will be provided with a movie description, and you will output a json object containing the following information:

{
    categories: string[] // Array of categories based on the movie description,
    summary: string // 1-sentence summary of the movie based on the movie description
}

Categories refer to the genre or type of the movie, like "action", "romance", "comedy", etc. Keep category names simple and use only lower case letters.
Movies can have several categories, but try to keep it under 3-4. Only mention the categories that are the most obvious based on the description.
'''

def get_categories(description):
    prompt = f"{categorize_system_prompt}\n\nMovie description:\n{description}\n\nRespond with the JSON only."

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "phi",
            "prompt": prompt,
            "temperature": 0.1,
            "stream": False
        }
    )
    response.raise_for_status()
    content = response.json().get("response", "")

    try:
        # Attempt to parse the returned JSON string to Python dict
        result = json.loads(content)
    except json.JSONDecodeError:
        print("Warning: Failed to parse JSON response:")
        print(content)
        result = None

    return result


In [50]:
import requests

def get_caption(img_url, title):
    caption_system_prompt = "Your system prompt here"

    prompt = f"{caption_system_prompt}\nTitle: {title}\nImage URL: {img_url}"

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "phi",
            "prompt": prompt,
            "temperature": 0.2,
            "max_tokens": 300,
            "stream": False
        }
    )
    response.raise_for_status()
    return response.json().get("response", "")


### Creating the batch job

As with the first example, we will create an array of json tasks to generate a `jsonl` file and use it to create the batch job.

In [52]:
# Creating an array of json tasks

tasks = []

for index, row in df.iterrows():
    
    title = row['title']
    img_url = row['primary_image']
    
    task = {
        "custom_id": f"task-{index}",
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            # This is what you would have in your Chat Completions API call
            "model": "gpt-4o-mini",
            "temperature": 0.2,
            "max_tokens": 300,
            "messages": [
                {
                    "role": "system",
                    "content": caption_system_prompt
                },
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": title
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": img_url
                            }
                        },
                    ],
                }
            ]            
        }
    }
    
    tasks.append(task)

In [53]:
# Creating the file

file_name = "data/batch_tasks_furniture.jsonl"

with open(file_name, 'w') as file:
    for obj in tasks:
        file.write(json.dumps(obj) + '\n')

In [54]:
import requests

with open(file_name, "r", encoding="utf-8") as f:
    file_content = f.read()

response = requests.post("http://localhost:11434/api/generate", json={
    "model": "phi",
    "prompt": file_content,
    "stream": False
})
response.raise_for_status()
result = response.json().get("response", "")


In [55]:
# Creating the job

import requests

def run_batch(prompts):
    results = []
    for prompt in prompts:
        response = requests.post("http://localhost:11434/api/generate", json={
            "model": "phi",
            "prompt": prompt,
            "stream": False
        })
        response.raise_for_status()
        results.append(response.json().get("response", ""))
    return results

# Example usage
prompts = [
    "Explain reinforcement learning in simple words.",
    "Describe the plot of Inception."
]

outputs = run_batch(prompts)
for output in outputs:
    print(output)


 Sure, I'd be happy to help! Reinforcement learning is when a computer program learns how to do something by getting rewarded or punished for its actions. It's like training a dog to fetch a ball. When the dog picks up the ball and brings it back, they get a treat. This is a reward that encourages them to keep doing what they did right. On the other hand, if the dog bites the ball instead of bringing it back, this could be seen as a punishment, which discourages the dog from biting in the future.

In the same way, when you teach an artificial intelligence program how to do something (like driving a car), you give it feedback based on whether or not its actions are correct. If the program takes the right turn at a traffic light and avoids hitting another vehicle, this is a positive reward that encourages the program to continue making the right decisions. However, if it runs a red light or crashes into another car, this is a negative reward that discourages the program from repeating th

In [56]:
# Assuming you have a list of prompts to process
prompts = ["Prompt 1", "Prompt 2", "Prompt 3"]
results = []

import requests

for prompt in prompts:
    response = requests.post("http://localhost:11434/api/generate", json={
        "model": "phi",
        "prompt": prompt,
        "stream": False
    })
    response.raise_for_status()
    results.append(response.json().get("response", ""))

# Now print all results


### Getting results

As with the first example, we can retrieve results once the batch job is done.

Reminder: the results are not in the same order as in the input file.
Make sure to check the custom_id to match the results against the input requests

In [58]:
# Retrieving result file

for i, result in enumerate(results):
    print(f"Result {i+1}:\n{result}\n")


Result 1:
 

Result 2:
 

Result 3:
 Prompt 3: Can you suggest some outdoor activities that we can do on a sunny day at the beach?




In [59]:
file_name = "data/batch_tasks_furniture.jsonl"

with open(file_name, 'w') as file:
    for obj in tasks:
        file.write(json.dumps(obj) + '\n')

In [73]:
import json

results = []
with open(result_file_name, 'r', encoding='utf-8') as file:
    for line in file:
        line = line.strip()
        if line:  # skip empty lines
            json_object = json.loads(line)
            results.append(json_object)


In [81]:
from IPython.display import Image, display

for res in results[:5]:
    try:
        task_id = res['custom_id']
        index = int(task_id.split('-')[-1])
        result = res['response']['body']['choices'][0]['message']['content']
        
        item = df.iloc[index]
        img_url = item['primary_image']
        
        img = Image(url=img_url)
        display(img)
        print(f"CAPTION: {result}\n\n")
    except (KeyError, IndexError, ValueError) as e:
        print(f"Skipping result due to error: {e}")


## Wrapping up

In this cookbook, we have seen two examples of how to use the new Batch API, but keep in mind that the Batch API works the same way as the Chat Completions endpoint, supporting the same parameters and most of the recent models (gpt-4o, gpt-4o-mini, gpt-4-turbo, gpt-3.5-turbo...).

By using this API, you can significantly reduce costs, so we recommend switching every workload that can happen async to a batch job with this new API.